In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv("fundamentals_dataset.csv")
df = df.head(50000)
df = df.dropna()
df.head()

In [ ]:
!pip install pandas sentence-transformers faiss-cpu pypdf

In [ ]:
def row_to_text(row):
    return f"In {row['period']}, {row['company']} reported {row['indicator']} of {row['amount']} {row['unit']}."

texts = df.apply(row_to_text, axis=1).tolist()

print(texts[:3])

In [ ]:
from pypdf import PdfReader

pdf_texts = []

for filename in uploaded.keys():
    if filename.endswith(".pdf"):
        reader = PdfReader(filename)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""
        pdf_texts.append(text)

print("PDFs processed:", len(pdf_texts))

In [ ]:
import re

def clean_text(text):
    return re.sub(r'\s+', ' ', text)

def chunk_text(text, chunk_size=300):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunks.append(" ".join(words[i:i+chunk_size]))
    return chunks

pdf_chunks = []

for text in pdf_texts:
    clean = clean_text(text)
    chunks = chunk_text(clean)
    pdf_chunks.extend(chunks)

print("Chunks created:", len(pdf_chunks))

In [ ]:
rules = [
"An emergency fund should cover 3 to 6 months of expenses.",
"The 50-30-20 rule divides income into needs, wants, and savings.",
"Avoid high-interest debt whenever possible.",
"Invest according to risk tolerance and time horizon.",
"Diversification reduces investment risk.",
"Track expenses regularly to improve budgeting.",
"Savings should be prioritized before discretionary spending.",
"Maintain a balance between liquidity and investment."
]

In [ ]:
all_texts = texts + pdf_chunks + rules

print("Total documents:", len(all_texts))
print("\nSample:\n", all_texts[:3])

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(all_texts, show_progress_bar=True)

In [ ]:
import faiss
import numpy as np

dimension = len(embeddings[0])

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("FAISS index created with", index.ntotal, "vectors")

In [ ]:
def search(query, k=5):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, k)

    results = [all_texts[i] for i in indices[0]]
    return results

In [ ]:
results = search("How should I save money every month?")

for r in results:
    print(r)
    print("----")

In [ ]:
def generate_answer(query):
    context = "\n".join(search(query, k=5))

    prompt = f"""
You are a financial advisor.

Using the context below, answer the user's question clearly.

Context:
{context}

Question:
{query}

Give:
1. Clear financial advice
2. Short explanation
3. Bullet points if needed
"""

    return prompt

In [ ]:
print(generate_answer("How should I save money every month?"))

In [ ]:
!pip install openai

In [ ]:
from openai import OpenAI
client = OpenAI(api_key="api-key")

def generate_answer(query):
    results = search(query, k=5)

    answer = "Financial Advice:\n\n"

    for r in results[:3]:
        answer += "- " + r[:200] + "\n\n"

    answer += "\nSummary:\n"
    answer += "Focus on budgeting, saving regularly, reducing unnecessary expenses, and avoiding debt."

    return answer

In [ ]:
print(generate_answer("How should I manage my monthly income?"))

In [ ]:
def generate_answer(query):
    results = search(query, k=5)

    advice_points = []

    for r in results:
        r = r.strip()

        # pick meaningful sentences only
        if len(r) > 50:
            advice_points.append(r[:150])

        if len(advice_points) == 3:
            break

    answer = "Financial Advice\n\n"

    for point in advice_points:
        answer += f"- {point}...\n\n"

    answer += "Explanation\n\n"
    answer += "Managing your money effectively requires tracking income, controlling expenses, and saving consistently. By following structured budgeting and avoiding unnecessary debt, you can build long-term financial stability.\n\n"

    answer += "Key Principle\n\n"
    answer += "Start saving early, spend wisely, and review your finances regularly."

    return answer

In [ ]:
print(generate_answer("How should I manage my monthly income?"))

In [ ]:
def get_user_profile():
    income = int(input("Enter monthly income: "))
    expenses = int(input("Enter monthly expenses: "))
    age = int(input("Enter age: "))
    goal = input("Enter financial goal (saving/investment/etc): ")

    return {
        "income": income,
        "expenses": expenses,
        "age": age,
        "goal": goal
    }

In [ ]:
def calculate_risk(profile):
    savings = profile["income"] - profile["expenses"]

    if savings < 10000:
        return "Low"
    elif savings < 30000:
        return "Moderate"
    else:
        return "High"

In [ ]:
def generate_personalized_answer(query, profile):
    results = search(query, k=5)

    risk = calculate_risk(profile)
    savings = profile["income"] - profile["expenses"]

    advice = "💡 Financial Advice\n\n"

    for r in results[:3]:
        advice += f"- {r[:150]}...\n\n"

    advice += "📊 Your Profile\n\n"
    advice += f"- Income: ₹{profile['income']}\n"
    advice += f"- Expenses: ₹{profile['expenses']}\n"
    advice += f"- Savings: ₹{savings}\n"
    advice += f"- Risk Level: {risk}\n\n"

    advice += "Personalized Insight\n\n"

    if risk == "Low":
        advice += "Focus on reducing expenses and building an emergency fund before investing.\n\n"
    elif risk == "Moderate":
        advice += "You can balance savings and start low-risk investments.\n\n"
    else:
        advice += "You can consider higher-return investments while maintaining savings discipline.\n\n"

    advice += "⚡ Key Principle\n\nStart saving early, spend wisely, and review finances regularly."

    return advice

In [ ]:
profile = get_user_profile()

print(generate_personalized_answer(
    "How should I manage my monthly income?",
    profile
))